# Sampling Selection for Manual Annotation

This notebook performs stratified sampling of comments to create a training dataset for a classification model.

## Annotation Process Context

**First Iteration (600 comments)**:
- 200 comments mentioning symptoms
- 200 comments mentioning substances (anabolic names)
- 200 comments with co-occurrence (symptoms + substances)
- **Result**: Cohen's Kappa = 0.6 (moderate agreement between annotators)

**Second Iteration (3000 comments)** - This notebook:
- 1000 comments mentioning symptoms
- 1000 comments mentioning substances
- 1000 comments with co-occurrence
- Disjoint sets (no overlap) using random sampling
- First iteration comments may still be present (due to suboptimal Kappa)

**Goal**: Create a balanced dataset to train a model that classifies comments into:
- **Self-report**: Personal reports of use
- **Symptoms/Side-effects**: Mentions of adverse effects

In [ ]:
import pandas as pd
import re
import os

# Project paths
PROJECT_ROOT = os.path.abspath(os.getcwd())
CLEANED_DATA_DIR = os.path.join(PROJECT_ROOT, 'cleaned_data')

In [ ]:
df = pd.read_csv(os.path.join(CLEANED_DATA_DIR, 'final_comments_match_cleaned.csv'))

In [ ]:
list_terms = ['durateston', 'masteron', 'testo', 'testosterona', 'trembolona', 'oxandrolona', 'oxan', 'primabolan', 'stano', 'hemogenin', 'dianabol', 'trenbolone', 'deca-durabolin', 'boldenona', 'winstrol', 'turinabol', 'proviron', 'halotestin', 'andriol', 'SARMs', 'aplicação de testo', 'ciclo de anabolizante', 'decanoato de nandrolona', 'enantato de testosterona', 'propionato de testosterona', 'cipionato de testosterona', 'undecilenato de boldenona', 'trembo ace']

list_symptoms = {
    "acne": ["espinhas", "erupções cutâneas", "inflamação na pele"],
    "agressividade": ["comportamento agressivo", "explosões de raiva"],
    "alterações no apetite": ["aumento do apetite", "perda de apetite", "distúrbios alimentares"],
    "alterações no fígado": ["lesão hepática", "disfunção hepática", "problemas hepáticos"],
    "alucinações": ["visões", "percepções irreais"],
    "ansiedade": ["preocupação excessiva", "inquietação"],
    "apneia do sono": ["paradas respiratórias durante o sono", "ronco"],
    "atrofia testicular": ["redução do volume testicular", "testículos encolhidos"],
    "aumento da próstata": ["hiperplasia prostática", "próstata aumentada"],
    "aumento de pelos corporais": ["hirsutismo", "crescimento excessivo de pelos"],
    "batimentos irregulares": ["arritmia", "palpitações"],
    "clitoromegalia": ["aumento do clitóris"],
    "colesterol alto": ["hipercolesterolemia"],
    "compulsão": ["uso compulsivo", "desejo incontrolável"],
    "cãibras musculares": ["espasmos musculares"],
    "danos nos rins": ["lesão renal", "nefropatia", "insuficiência renal"],
    "dependência": ["uso contínuo", "não conseguir parar"],
    "depressão": ["tristeza profunda", "humor deprimido", "apatia"],
    "dificuldade para urinar": ["jato fraco", "retenção urinária"],
    "disfunção erétil": ["impotência", "dificuldade de ereção"],
    "distúrbios menstruais": ["menstruação irregular", "amenorreia"],
    "dor de cabeça": ["cefaleia"],
    "dor/inflamação no local da injeção": ["dor no local da aplicação",  "inflamação no local da aplicação", "dor no local da aplicação", "inchaço na aplicação"],
    "dores abdominais": ["dor na barriga", "desconforto abdominal"],
    "dores musculares": ["mialgia"],
    "engrossamento da voz": ["voz mais grave", "voz profunda"],
    "euforia": ["sensação de poder", "excesso de excitação"],
    "fadiga": ["cansaço extremo", "exaustão"],
    "ginecomastia": ["aumento das mamas", "sensibilidade nos mamilos"],
    "hipertrofia": ["crescimento muscular"],
    "hipogonadismo": ["baixa testosterona", "disfunção hormonal masculina"],
    "hipotireoidismo": ["função tireoidiana reduzida"],
    "icterícia": ["pele amarelada", "olhos amarelados"],
    "impotência": ["disfunção erétil", "problemas de ereção"],
    "infarto": ["ataque cardíaco", "infarto do miocárdio"],
    "infertilidade": ["baixa fertilidade", "diminuição da produção de esperma"],
    "insuficiência renal": ["falência renal", "doença renal"],
    "insônia": ["dificuldade para dormir", "sono interrompido"],
    "irritabilidade": ["facilidade para se irritar", "mau humor"],
    "letargia": ["apatia", "lentidão"],
    "mania": ["hiperatividade", "excesso de energia"],
    "náuseas": ["enjoo", "vontade de vomitar"],
    "oleosidade na pele": ["pele oleosa"],
    "oscilações de humor": ["mudanças de humor", "humor instável"],
    "osteoporose": ["ossos frágeis", "redução da densidade óssea"],
    "paranoia": ["desconfiança excessiva", "medo irracional"],
    "pele oleosa": ["brilho excessivo na pele"],
    "pele oleosa excessiva": ["seborréia"],
    "perda de apetite": ["falta de fome", "anorexia"],
    "perda de libido": ["baixa libido", "redução do desejo sexual"],
    "policitemia": ["aumento dos glóbulos vermelhos"],
    "pressão alta": ["hipertensão"],
    "priapismo": ["ereção prolongada"],
    "problemas cardiovasculares": ["doenças do coração", "complicações cardíacas"],
    "psicose": ["quebra da realidade", "delírios"],
    "puberdade precoce": ["maturação sexual antecipada"],
    "queda de cabelo": ["alopecia", "calvície"],
    "retenção de líquido": ["inchaço", "edema"],
    "suor excessivo": ["hiperidrose", "suor noturno"],
    "toxicidade hepática": ["dano hepático", "hepatite medicamentosa"],
    "tremores": ["movimentos involuntários"],
    "trombose": ["coágulos sanguíneos"],
    "tumores hepáticos": ["neoplasias hepáticas", "câncer no fígado"],
    "voz grossa": ["voz masculina", "mudança vocal"],
    "vômitos": ["emese"]
}

---
## 1. Definition of Terms and Symptoms

Curated lists of:
- **list_terms**: Names of anabolic substances and related compounds (durateston, oxandrolone, SARMs, etc.)
- **list_symptoms**: Dictionary of symptoms/side-effects with linguistic variations (e.g., "ginecomastia" includes "breast enlargement", "nipple sensitivity")

These lists are used to identify relevant comments via regex pattern matching.

In [ ]:
all_symptoms = []
for main, variations in list_symptoms.items():
    all_symptoms.append(main)
    all_symptoms.extend(variations)

pattern_symptoms = '|'.join([r'\b' + re.escape(symptom) + r'\b' for symptom in all_symptoms])
mask_symptoms = df['comment'].str.contains(pattern_symptoms, case=False, na=False)

pattern_terms = '|'.join([r'\b' + re.escape(term) + r'\b' for term in list_terms])
mask_terms = df['comment'].str.contains(pattern_terms, case=False, na=False)


---
## 2. Comment Filtering

Create boolean masks to identify comments that mention:
- **mask_symptoms**: Any symptom from the list (main terms + variations)
- **mask_terms**: Any anabolic substance from the list
- **Co-occurrence**: Comments that mention both (symptoms AND substances)

We use regex with `\b` (word boundaries) to avoid partial matches.

In [ ]:
df = df[['comment_id', 'comment']]

df_symptoms = df[mask_symptoms]
df_terms = df[mask_terms]
df_cooccurrence = df[mask_symptoms & mask_terms]

In [ ]:
print(f'Total comments mentioning terms: {df_terms.shape[0]}')
print(f'Total comments mentioning symptoms: {df_symptoms.shape[0]}')
print(f'Total comments with co-occurrence: {df_cooccurrence.shape[0]}')

In [ ]:
cooc_sample = df_cooccurrence.sample(n=1000, random_state=42)

# Remove sampled IDs from other groups
DF_SYMPTOMS = df_symptoms[~df_symptoms['comment_id'].isin(cooc_sample['comment_id'])]
DF_TERMS = df_terms[~df_terms['comment_id'].isin(cooc_sample['comment_id'])]

symptoms_sample = DF_SYMPTOMS.sample(n=1000, random_state=42)

# Remove sampled IDs from symptoms out of terms
DF_TERMS = DF_TERMS[~DF_TERMS['comment_id'].isin(symptoms_sample['comment_id'])]

terms_sample = DF_TERMS.sample(n=1000, random_state=42)

---
## 3. Disjoint Stratified Sampling

Sequential process to ensure non-overlapping sets:
1. **Co-occurrence sample** (1000): Randomly sample from intersection
2. **Remove IDs** of co-occurrence from the other two sets
3. **Symptoms sample** (1000): Sample from remaining comments
4. **Remove IDs** of symptoms from the terms set
5. **Terms sample** (1000): Sample from remaining comments

**Result**: 3000 unique disjoint comments (random_state=42 for reproducibility).

In [ ]:
cooc_sample = cooc_sample.reset_index(drop=True)
symptoms_sample = symptoms_sample.reset_index(drop=True)
terms_sample = terms_sample.reset_index(drop=True)

In [ ]:
df_annotations = pd.concat([cooc_sample, symptoms_sample, terms_sample], ignore_index=True)

In [ ]:
df_annotations = df_annotations.sample(frac=1, random_state=42).reset_index(drop=True)

---
## 4. Final Preparation

- **Concatenation**: Combine the 3 samples into a single dataset
- **Shuffle**: Shuffle comments to avoid ordering bias during annotation
- **Validation**: Check for duplicate IDs
- **Output**: Dataset ready to distribute to annotators (`data_to_label.csv`)

In [ ]:
df_annotations.head()

In [ ]:
df_annotations.shape

In [ ]:
df_annotations['comment_id'].duplicated().sum()

In [ ]:
# df_annotations.to_csv(os.path.join(PROJECT_ROOT, 'annotation', 'data_to_label.csv'), index=False)